<a href="https://colab.research.google.com/github/d-vf/phd-website-content/blob/main/assets/scholar_scrapping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get update -qq
!apt-get install -y chromium chromium-driver -qq
!pip install selenium beautifulsoup4 pandas -q

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
E: Package 'chromium' has no installation candidate


In [ ]:
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google.list


!apt-get update -qq
!apt-get install -y google-chrome-stable -qq

!pip install -q selenium webdriver-manager pandas bs4

OK
W: http://dl.google.com/linux/chrome/deb/dists/stable/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libatk1.0-data.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../00-libatk1.0-data_2.36.0-3build1_all.deb ...
Unpacking libatk1.0-data (2.36.0-3build1) ...
Selecting previously unselected package libatk1.0-0:amd64.
Preparing to unpack .../01-libatk1.0-0_2.36.0-3build1_amd64.deb ...
Unpacking libatk1.0-0:amd64 (2.36.0-3build1) ...
Selecting previously unselected package libatspi2.0-0:amd64.
Preparing to unpack .../02-libatspi2.0-0_2.44.0-3_amd64.deb ...
Unpacking libatspi2.0-0:amd64 (2.44.0-3) ...
Selecting previously unselecte

In [ ]:
import pandas as pd
import time
import re
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

# Limit of records per professor
MAX_RECORDS = 5

# Robust Chrome Configuration for Colab
options = Options()
options.add_argument('--headless=new')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--disable-gpu')

print("A iniciar o navegador e a extrair os dados...\n")
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)

professors = {
    "Ana Paula Ferreira Dias Barbosa Póvoa": "https://scholar.tecnico.ulisboa.pt/authors/ist13662/supervised-records?lang=pt&type=doctoral-thesis",
    "Rui Miguel Loureiro Nobre Baptista": "https://scholar.tecnico.ulisboa.pt/authors/ist14021/supervised-records?lang=pt&type=doctoral-thesis",
    "Margarida Catalão Lopes": "https://scholar.tecnico.ulisboa.pt/authors/ist14105/supervised-records?lang=pt&type=doctoral-thesis",
    "Mónica Duarte Correia de Oliveira": "https://scholar.tecnico.ulisboa.pt/authors/ist14410/supervised-records?lang=pt&type=doctoral-thesis",
    "José Rui de Matos Figueira": "https://scholar.tecnico.ulisboa.pt/authors/ist14525/supervised-records?lang=pt&type=doctoral-thesis",
    "Joana Serra da Luz Mendonça": "https://scholar.tecnico.ulisboa.pt/authors/ist45229/supervised-records?lang=pt&type=doctoral-thesis",
    "Francisco Lima": "https://scholar.tecnico.ulisboa.pt/authors/ist14367/supervised-records?lang=pt&type=doctoral-thesis",
    "Ana Isabel Cerqueira de Sousa Gouveia Carvalho": "https://scholar.tecnico.ulisboa.pt/authors/ist149893/supervised-records?lang=pt&type=doctoral-thesis",
    "Tânia Rute Xavier de Matos Pinto-Varela": "https://scholar.tecnico.ulisboa.pt/authors/ist25305/supervised-records?lang=pt&type=doctoral-thesis",
    "Miguel Torres Preto": "https://scholar.tecnico.ulisboa.pt/authors/ist45281/supervised-records?lang=pt&type=doctoral-thesis",
    "Susana Isabel Carvalho Relvas": "https://scholar.tecnico.ulisboa.pt/authors/ist46455/supervised-records?lang=pt&type=doctoral-thesis",
    "Tânia Rodrigues Pereira Ramos": "https://scholar.tecnico.ulisboa.pt/authors/ist46496/supervised-records?lang=pt&type=doctoral-thesis",
    "Carlos Manuel Ferreira Monteiro": "https://scholar.tecnico.ulisboa.pt/authors/ist12228/supervised-records?lang=pt&type=doctoral-thesis",
    "Maria Isabel Craveiro Pedro": "https://scholar.tecnico.ulisboa.pt/authors/ist13156/supervised-records?lang=pt&type=doctoral-thesis",
    "João Carlos da Cruz Lourenço": "https://scholar.tecnico.ulisboa.pt/authors/ist14341/supervised-records?lang=pt&type=doctoral-thesis",
    "Hugo Miguel Fragoso de Castro Silva": "https://scholar.tecnico.ulisboa.pt/authors/ist152309/supervised-records?lang=pt&type=doctoral-thesis",
    "António Sérgio Constantino Folgado Ribeiro": "https://scholar.tecnico.ulisboa.pt/authors/ist154299/supervised-records?lang=pt&type=doctoral-thesis",
    "Bruna Alexandra Elias Mota": "https://scholar.tecnico.ulisboa.pt/authors/ist155745/supervised-records?lang=pt&type=doctoral-thesis",
    "Cátia Rafaela Ferreira Medeiros da Silva": "https://scholar.tecnico.ulisboa.pt/authors/ist165165/supervised-records?lang=pt&type=doctoral-thesis",
    "Daniel Rebelo dos Santos": "https://scholar.tecnico.ulisboa.pt/authors/ist167904/supervised-records?lang=pt&type=doctoral-thesis",
    "Diana Rita Ramos Jorge": "https://scholar.tecnico.ulisboa.pt/authors/ist172032/supervised-records?lang=pt&type=doctoral-thesis",
    "Miguel Alves Pereira": "https://scholar.tecnico.ulisboa.pt/authors/ist176052/supervised-records?lang=pt&type=doctoral-thesis",
    "Ana Lopes Vieira": "https://scholar.tecnico.ulisboa.pt/authors/ist177517/supervised-records?lang=pt&type=doctoral-thesis",
    "Rafael Alexandre Pires Miranda": "https://scholar.tecnico.ulisboa.pt/authors/ist178586/supervised-records?lang=pt&type=doctoral-thesis",
    "Inês Isabel Carrilho Nunes": "https://scholar.tecnico.ulisboa.pt/authors/ist423260/supervised-records?lang=pt&type=doctoral-thesis",
    "Inês Marques Proença": "https://scholar.tecnico.ulisboa.pt/authors/ist427860/supervised-records?lang=pt&type=doctoral-thesis",
    "Celso Augusto de Matos": "https://scholar.tecnico.ulisboa.pt/authors/ist429782/supervised-records?lang=pt&type=doctoral-thesis",
    "Sérgio Filipe Assunção Batista": "https://scholar.tecnico.ulisboa.pt/authors/ist430700/supervised-records?lang=pt&type=doctoral-thesis",
    "João Manuel Jorge Estêvão": "https://scholar.tecnico.ulisboa.pt/authors/ist430772/supervised-records?lang=pt&type=doctoral-thesis",
    "António Miguel Areias Dias Amaral": "https://scholar.tecnico.ulisboa.pt/authors/ist45356/supervised-records?lang=pt&type=doctoral-thesis",
    "Carla Maria do Rosário Costa": "https://scholar.tecnico.ulisboa.pt/authors/ist90190/supervised-records?lang=pt&type=doctoral-thesis",
    "Miguel Leitão Bignolas Mira da Silva": "https://scholar.tecnico.ulisboa.pt/authors/ist13948/supervised-records?lang=pt&type=doctoral-thesis",
    "Carlos Augusto Santos Silva": "https://scholar.tecnico.ulisboa.pt/authors/ist23960/supervised-records?lang=pt&type=doctoral-thesis",
    "Patrícia De Carvalho Baptista": "https://scholar.tecnico.ulisboa.pt/authors/ist151313/supervised-records?lang=pt&type=doctoral-thesis",
    "Manuel Frederico Tojal de Valsassina Heitor": "https://scholar.tecnico.ulisboa.pt/authors/ist12370/supervised-records?lang=pt&type=doctoral-thesis",
    "Célia Maria Santos Cardoso de Jesus ": "https://scholar.tecnico.ulisboa.pt/authors/ist12937/supervised-records?lang=pt&type=doctoral-thesis",
    "Pedro Manuel Santos de Carvalho ": "https://scholar.tecnico.ulisboa.pt/authors/ist13407/supervised-records?lang=pt&type=doctoral-thesis",
    "João Paulo Salgado Arriscado Costeira": "https://scholar.tecnico.ulisboa.pt/authors/ist12390/supervised-records?lang=pt&type=doctoral-thesis"

}

extracted_data = []

for name, url in professors.items():
    try:
        driver.get(url)
        # Esperar 4 segundos pelo conteúdo dinâmico
        time.sleep(4)

        soup = BeautifulSoup(driver.page_source, 'html.parser')

        # Isolar a área principal
        main_content = soup.find('main') or soup.find('div', id='main') or soup
        entries = main_content.find_all('li')

        count = 0
        for entry in entries:
            if count >= MAX_RECORDS:
                break

            links = entry.find_all('a')
            if not links:
                continue

            title_tag = links[0]
            title = title_tag.get_text(strip=True)

            # Filtrar menus indesejados
            if any(word in title for word in ["Entrar", "Idioma", "Voltar", "Menu", "Sign in"]):
                continue

            if len(title) < 4:
                continue

            text = entry.get_text(separator=" ", strip=True)

            # === CORREÇÃO DO ANO: Adicionado ?: para capturar os 4 dígitos ===
            year_matches = re.findall(r'\b(?:19|20)\d{2}\b', text)
            if not year_matches:
                continue
            year = year_matches[-1]

            link = title_tag.get('href', '')
            if link.startswith('/'):
                link = "https://scholar.tecnico.ulisboa.pt" + link

            raw_info = text.replace(title, "", 1).strip()
            raw_info = re.sub(r'^[\W_]+', '', raw_info).strip()

            student = "Desconhecido"
            parts = re.split(r'[—–\-•·|]', raw_info)
            if len(parts) > 0 and len(parts[0].strip()) > 2:
                student = parts[0].strip()
            else:
                student = raw_info.replace(year, "").strip()

            # Remover o prefixo "Tese de Doutoramento" e variações
            unwanted_labels = [
                "Tese de Doutoramento",
                "Doctoral Thesis",
                "PhD Thesis",
                "Tese de Mestrado",
                "Tese de"
            ]
            for label in unwanted_labels:
                student = student.replace(label, "")

            # Limpeza final de pontuação no início do nome
            student = re.sub(r'^[\W_]+', '', student).strip()
            student = student[:60]

            extracted_data.append({
                "Professor / Supervisor": name,
                "PhD Student": student,
                "Year": year,
                "Thesis Title": title,
                "Direct Link": link
            })
            count += 1

        if count == 0:
            print(f"  --> Nenhuma tese encontrada para {name}.")
        else:
            print(f"  --> Sucesso: {count} teses extraídas para {name}.")

    except Exception as e:
        print(f"Erro ao processar {name}: {e}")

driver.quit()

df_results = pd.DataFrame(extracted_data)
display(df_results)
df_results.to_csv("phd_theses_final.csv", index=False, encoding='utf-8-sig')
print(f"\nConcluído! Ficheiro 'phd_theses_final.csv' gerado com sucesso com {len(df_results)} registos.")

A iniciar o navegador e a extrair os dados...

  --> Sucesso: 5 teses extraídas para Ana Paula Ferreira Dias Barbosa Póvoa.
  --> Sucesso: 5 teses extraídas para Rui Miguel Loureiro Nobre Baptista.
  --> Sucesso: 3 teses extraídas para Margarida Catalão Lopes.
  --> Sucesso: 5 teses extraídas para Mónica Duarte Correia de Oliveira.
  --> Sucesso: 5 teses extraídas para José Rui de Matos Figueira.
  --> Sucesso: 5 teses extraídas para Joana Serra da Luz Mendonça.
  --> Sucesso: 4 teses extraídas para Francisco Lima.
  --> Sucesso: 4 teses extraídas para Ana Isabel Cerqueira de Sousa Gouveia Carvalho.
  --> Sucesso: 1 teses extraídas para Tânia Rute Xavier de Matos Pinto-Varela.
  --> Sucesso: 1 teses extraídas para Miguel Torres Preto.
  --> Sucesso: 5 teses extraídas para Susana Isabel Carvalho Relvas.
  --> Sucesso: 3 teses extraídas para Tânia Rodrigues Pereira Ramos.
  --> Nenhuma tese encontrada para Carlos Manuel Ferreira Monteiro.
  --> Nenhuma tese encontrada para Maria Isabel C

,Professor / Supervisor,PhD Student,Year,Thesis Title,Direct Link
0,Ana Paula Ferreira Dias Barbosa Póvoa,Ana Sofia Bernardo Torrado,2025,Sustainable blood supply chain optimization: a...,https://scholar.tecnico.ulisboa.pt/records/RJT...
1,Ana Paula Ferreira Dias Barbosa Póvoa,Maria Odete de Oliveira Meneses,2025,Enhancing blood supply chain management: integ...,https://scholar.tecnico.ulisboa.pt/records/YiR...
2,Ana Paula Ferreira Dias Barbosa Póvoa,Mariana Bayão Horta Mesquita da Cunha e Silva,2024,Advancing Multi-Objective Optimization: Repres...,https://scholar.tecnico.ulisboa.pt/records/vC3...
3,Ana Paula Ferreira Dias Barbosa Póvoa,João Henrique Pires Ribeiro,2024,Quantitative tools to aid decision-making towa...,https://scholar.tecnico.ulisboa.pt/records/N85...
4,Ana Paula Ferreira Dias Barbosa Póvoa,Paulo Roberto de Sousa Abreu,2023,A data-driven optimization approach for resour...,https://scholar.tecnico.ulisboa.pt/records/g7L...
...,...,...,...,...,...
81,João Paulo Salgado Arriscado Costeira,Stevo Rackovic,2025,Mathematical methods for inverse rigging in re...,https://scholar.tecnico.ulisboa.pt/records/Ngk...
82,João Paulo Salgado Arriscado Costeira,Jude Mukundane,2022,Augmented communication technologies for dista...,https://scholar.tecnico.ulisboa.pt/records/2-J...
83,João Paulo Salgado Arriscado Costeira,Fábio Rúben Silva Mendonça,2021,Signal processing approaches for sleep quality...,https://scholar.tecnico.ulisboa.pt/records/tCC...
84,João Paulo Salgado Arriscado Costeira,Maria Beatriz Alves de Sousa Quintino Ferreira,2021,Classification of visual data with unreliable ...,https://scholar.tecnico.ulisboa.pt/records/GaQ...



Concluído! Ficheiro 'phd_theses_final.csv' gerado com sucesso com 86 registos.


In [ ]:
 !pip install tabulate

In [ ]:
print(df_results.to_markdown(index=False))

| Professor / Supervisor                         | PhD Student                                      |   Year | Thesis Title                                                                                                                                                                                                        | Direct Link                                                                     |
|:-----------------------------------------------|:-------------------------------------------------|-------:|:--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|:--------------------------------------------------------------------------------|
| Ana Paula Ferreira Dias Barbosa Póvoa          | Ana Sofia Bernardo Torrado                       |   2025 | Sustainable blood supply chain optimization: addressing demand and supply, 